# QF600 Asset Pricing — Backtesting a Portfolio Strategy & Computing Alpha

Backtest an **investor portfolio** against a **benchmark portfolio**, both rebalanced on a fixed
schedule, then decompose the result into the part explained by market exposure and the part that is not:

$$R_p - R_f \;=\; \alpha \;+\; \beta\,(R_m - R_f) \;+\; \varepsilon$$


Documentation in this [Github link](https://github.com/cyee02/MQF/tree/main/QF600/Assignment%201%20-%20Compute%20Alpha)

---
## 1. Set up

All third-party libraries are declared here and nowhere else in the notebook.

In [125]:
# Environment bootstrap: install anything missing, then switch on an interactive table viewer.
import importlib
import subprocess
import sys


def ensure_installed(package: str, module: str | None = None) -> None:
    """pip-install `package` only if `module` cannot already be imported.

    Parameters
    ----------
    package : str         name to pass to `pip install`
    module  : str | None  import name, when it differs from `package` (e.g. "lets_plot"
                          for "lets-plot"); defaults to `package`

    Returns
    -------
    None  called for its side effect: the package is importable afterwards
    """
    try:
        importlib.import_module(module or package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


ensure_installed("yfinance")
ensure_installed("lets-plot", "lets_plot")
ensure_installed("tabulate")            # pandas needs it for DataFrame.to_markdown() in §8

try:
    import google.colab                                              # noqa: F401
    IN_COLAB = True
    get_ipython().run_line_magic("load_ext", "google.colab.data_table")  # paginated tables in Colab
except ImportError:
    IN_COLAB = False                                                 # local -> Data Wrangler plugin

print(f"Running in Colab: {IN_COLAB}")

Running in Colab: False


In [126]:
import datetime as dt

import numpy as np
import pandas as pd
import yfinance as yf
from scipy import stats

from IPython.display import Markdown, display
from lets_plot import *
LetsPlot.setup_html()

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.6f}".format)
pd.set_option("display.max_rows", 10)

---
## 2. Parameters

Everything a user would want to change lives in this one cell.

The strategy horizon is expressed as a **start / end date** rather than a raw day count: the spec
fixes it at "01 Jan 2016", and pinning the dates avoids the calendar-days vs trading-days ambiguity.
The horizon *in trading days* is derived from the data further below.

`OPTIMISE_WEIGHTS` picks which of the two investor portfolios is backtested. Off, it is the fixed
`INVESTOR_PORTFOLIO` mix, held for the whole horizon. On, §4 re-solves the maximum-Sharpe mix at
every rebalance by Monte Carlo — `MC_RUNS` random long-only allocations, scored on the trailing
`LOOKBACK_DAYS` of returns. That window ends the day the block begins, so the weights a block trades
on are never fitted to the returns that block goes on to earn.

`RISK_FREE_TICKER` picks which risk-free convention every headline figure — Sharpe, CVaR in
dollars, the alpha/beta regression, §6.4's summary — is measured against: `"^TNX"` (10-year
Treasury), `"^FVX"` (5-year), or `None` (Rf = 0, i.e. raw returns with nothing subtracted). §3.3
computes all three regardless of this choice, so §6.3 and §7 can show how the alpha estimate
moves under each.

In [127]:
INVESTOR_PORTFOLIO = {
    "SPY": 0.5,  # Tracks NASDAQ-100 Index
    # "BAI": 0.25,  #  Managed Futures Strategy. With reference to top 20 Commodity Trading Advisor (CTA) hedge funds, the fund buys a smaller, cheaper group of about 10 to 15 liquid futures and forward contracts to match the hedge funds' returns
    # "IBIT": 0.25,  # SPDR Gold Shares ETF
    "USMV": 0.5,  # SPDR Gold Shares ETF
}

BENCHMARK_PORTFOLIO = {
    "SPY" : 1,          # S&P 500 ETF
}

START_DATE       = "2016-01-01"               # ~10 years of history
END_DATE         = dt.date.today()
REBALANCE_DAYS   = 60                         # trading days between rebalances (~1 quarter)
# REBALANCE_DAYS = int((END_DATE - dt.date.fromisoformat(START_DATE)).days * 0.7)  # No rebalance for now, just compute the returns of the portfolio over the entire period

OPTIMISE_WEIGHTS = False                       # True  -> re-solve the investor's max-Sharpe mix at
#          every rebalance (Markowitz MC, §4)
# False -> hold INVESTOR_PORTFOLIO for the whole horizon
MC_RUNS          = 10_000                      # Monte Carlo draws per rebalance
LOOKBACK_DAYS    = 252 * 5                    # trailing window the optimiser estimates from (~5 year)
RANDOM_SEED      = 42                         # fixes the draws, so a re-run reproduces the backtest

INITIAL_CAPITAL  = 100_000                    # USD at inception
TRADING_DAYS     = 252                        # annualisation factor
CVAR_CONFIDENCE  = 0.95                       # CVaR tail cutoff: average of the worst 5% of days

RISK_FREE_TICKER = "^TNX"                     # the convention every headline figure below uses:
# "^TNX" -> CBOE 10-year Treasury yield
# "^FVX" -> CBOE 5-year Treasury yield
# None   -> Rf = 0 (raw returns, nothing subtracted)
# §3.3 computes all three regardless, so §6.3/§7 can
# compare the alpha estimate across all of them.

# The investor weights are the mix held throughout when OPTIMISE_WEIGHTS is off, and both the
# investable universe and the early-block fallback when it is on. Either way they must be valid.
for name, portfolio in [("Investor", INVESTOR_PORTFOLIO), ("Benchmark", BENCHMARK_PORTFOLIO)]:
    assert np.isclose(sum(portfolio.values()), 1.0), f"{name} weights must sum to 1.0"

TICKERS = list(dict.fromkeys([*INVESTOR_PORTFOLIO, *BENCHMARK_PORTFOLIO]))   # de-duplicated
TICKERS

['SPY', 'USMV']

---
## 3. Data

### 3.1 Daily close prices

The download is kept twice: `prices_raw` exactly as returned, so §3.2 can measure what is
missing, and `prices` restricted to dates every ticker traded, which is what the backtest uses.

In [128]:
prices_raw =\
(
    yf.download(TICKERS,                # every ticker in one request
                start        = START_DATE,
                end          = END_DATE,
                auto_adjust  = True,    # adjust for splits & dividends -> total-return prices
                progress     = False
               )
    ["Close"]                           # daily close only
    [TICKERS]                           # restore the declared column order
)

prices =\
(
    prices_raw
    .dropna()                           # keep only dates every ticker traded
)

prices

Ticker,SPY,USMV
Date,,
2016-01-04,169.051758,34.031616
2016-01-05,169.337692,34.262333
2016-01-06,167.201584,34.048084
2016-01-07,163.190247,33.495998
2016-01-08,161.398956,33.158150
...,...,...
2026-10-12,778.099976,100.730003
2026-10-13,778.010010,100.650002
2026-10-14,769.599976,100.629997


### 3.2 Data completeness

Measured on `prices_raw`, before `.dropna()` hid anything. A ticker can be short of days for two
very different reasons, so they are counted separately:

- **a late start or early end** — the fund simply did not exist on those dates (`SOXX` and `GLD`
  both predate 2016, but this catches it for any ticker swapped into §2);
- **interior gaps** — a day inside the ticker's own coverage where every other ticker printed a
  price and this one did not. These are the ones worth worrying about.

Any date where even one ticker is missing is dropped from the backtest, so the shared calendar is
the intersection, not the union.

In [129]:
calendar_days = len(prices_raw)          # union calendar: any date at least one ticker traded

gaps_within_coverage =\
(
    prices_raw
    .apply(lambda column: column
                          .loc[column.first_valid_index():column.last_valid_index()]
                          .isna()
                          .sum())      # missing days strictly inside each ticker's own history
)

completeness =\
(
    pd.DataFrame({"First Quote"  : prices_raw.apply(lambda column: column.first_valid_index()),
                  "Last Quote"   : prices_raw.apply(lambda column: column.last_valid_index()),
                  "Trading Days" : calendar_days,
                  "Observed"     : prices_raw.count(),
                  "Missing"      : prices_raw.isna().sum(),
                  "Interior Gaps": gaps_within_coverage})
    .assign(**{"Missing %": lambda frame: (frame["Missing"] / calendar_days).map("{:.2%}".format)})
    .rename_axis("Ticker")
)

print(f"Union calendar : {prices_raw.index[0]:%d %b %Y} -> {prices_raw.index[-1]:%d %b %Y} "
      f"({calendar_days:,} trading days, the union across all tickers)")
print(f"Shared calendar: {len(prices):,} days kept by .dropna() "
      f"({calendar_days - len(prices):,} dropped, {1 - len(prices) / calendar_days:.2%})")

if gaps_within_coverage.any():
    print("Interior gaps (a ticker missing a day the others traded): "
          + ", ".join(f"{ticker} {count}"
                      for ticker, count in gaps_within_coverage[gaps_within_coverage > 0].items()))
else:
    print("No interior gaps: every ticker quotes a price on every day inside its own coverage.")

completeness

Union calendar : 04 Jan 2016 -> 16 Oct 2026 (2,713 trading days, the union across all tickers)
Shared calendar: 2,713 days kept by .dropna() (0 dropped, 0.00%)
No interior gaps: every ticker quotes a price on every day inside its own coverage.


,First Quote,Last Quote,Trading Days,Observed,Missing,Interior Gaps,Missing %
Ticker,,,,,,,
SPY,2016-01-04,2026-10-16,2713,2713,0,0,0.00%
USMV,2016-01-04,2026-10-16,2713,2713,0,0,0.00%


### 3.3 Risk-free rate

A CBOE Treasury yield index quotes its yield as an annualised **percent** (e.g. `2.27` = 2.27%).
Each is converted to a compounded daily rate and aligned to the trading calendar of `prices`, so
it can be subtracted from daily returns directly.

All three conventions in play throughout the rest of the notebook — 10-year (`^TNX`), 5-year
(`^FVX`), and Rf = 0 — are computed here unconditionally. `RISK_FREE_TICKER` (§2) only picks which
one is `rf_daily`, the series every other section uses; §6.3 and §7 use all three, to show how
sensitive the alpha estimate is to the risk-free assumption.

In [130]:
def yield_to_daily_rf(ticker: str) -> pd.Series:
    """A CBOE Treasury yield index, as a compounded daily rate aligned to `prices`.

    Parameters
    ----------
    ticker : str  a CBOE yield index, e.g. "^TNX" (10-year) or "^FVX" (5-year)

    Returns
    -------
    pd.Series  daily compounded rate, indexed like `prices`
    """
    return (
        yf.download(ticker,
                    start        = START_DATE,
                    end          = END_DATE,
                    auto_adjust  = True,
                    progress     = False
                   )
        ["Close"]
        .squeeze()                          # single-column frame -> Series
        .div(100)                           # percent -> decimal, annualised
        .add(1)
        .pow(1 / TRADING_DAYS)              # annual -> daily, compounded
        .sub(1)
        .reindex(prices.index)              # onto the equity trading calendar
        .ffill()                            # carry the last quote over yield-market holidays
        .bfill()                            # cover a missing first observation
    )


# All three risk-free conventions, computed unconditionally: RISK_FREE_TICKER (§2) only decides
# which one becomes `rf_daily` and feeds the rest of the notebook; §6.3 and §7 compare all three.
RF_CONVENTIONS = {
    "^TNX": ("10-Year Treasury", yield_to_daily_rf("^TNX")),
    "^FVX": ("5-Year Treasury",  yield_to_daily_rf("^FVX")),
    None  : ("Rf = 0",           pd.Series(0.0, index = prices.index)),
}

RF_LABEL, rf_daily = RF_CONVENTIONS[RISK_FREE_TICKER]
rf_daily = rf_daily.rename("rf_daily")

print(f"Risk-free convention in use: {RF_LABEL}")
rf_daily.to_frame()

Risk-free convention in use: 10-Year Treasury


,rf_daily
Date,
2016-01-04,0.000088
2016-01-05,0.000088
2016-01-06,0.000085
2016-01-07,0.000085
2016-01-08,0.000084
...,...
2026-10-12,0.000203
2026-10-13,0.000202
2026-10-14,0.000201


### 3.4 Daily asset returns and the derived horizon

Day 0 is inception, so its return is forced to `0` rather than dropped — this keeps every curve
starting at exactly `INITIAL_CAPITAL` on the first date.

In [131]:
asset_returns =\
(
    prices
    .pct_change()                       # daily simple returns
    .fillna(0)                          # day 0 = inception, no return yet
)

HORIZON_DAYS = len(asset_returns)       # horizon, in trading days

print(f"Horizon: {prices.index[0]:%d %b %Y} -> {prices.index[-1]:%d %b %Y} "
      f"({HORIZON_DAYS:,} trading days, {HORIZON_DAYS / TRADING_DAYS:.2f} years)")
print(f"Number of Rebalances over the horizon: {(HORIZON_DAYS - 1) // REBALANCE_DAYS}")

asset_returns

Horizon: 04 Jan 2016 -> 16 Oct 2026 (2,713 trading days, 10.77 years)
Number of Rebalances over the horizon: 45


Ticker,SPY,USMV
Date,,
2016-01-04,0.000000,0.000000
2016-01-05,0.001691,0.006779
2016-01-06,-0.012614,-0.006253
2016-01-07,-0.023991,-0.016215
2016-01-08,-0.010977,-0.010086
...,...,...
2026-10-12,-0.000604,0.001990
2026-10-13,-0.000116,-0.000794
2026-10-14,-0.010810,-0.000199


---
## 4. Reusable functions

Seven small building blocks, each doing one job: rebalance a portfolio, solve for the max-Sharpe
weights on a window, schedule those weights across the rebalance blocks, measure a drawdown,
measure tail risk, summarise risk & return, and regress one excess-return series on another.

In [132]:
w      = pd.Series(INVESTOR_PORTFOLIO, dtype = float)
w
blocks = pd.Series(np.arange(len(asset_returns)) // REBALANCE_DAYS,
                    index = asset_returns.index)

value_per_dollar =\
(
    asset_returns
    [w.index]                        # only this portfolio's assets, in weight order
    .add(1)
    .groupby(blocks)
    # .cumprod()                       # growth of $1 per asset, restarting each block
    # .mul(w, axis = "columns")        # weighted at the block's target weights
    # .sum(axis = "columns")           # -> portfolio value per $1 invested at block start
)
value_per_dollar

In [133]:
def do_rebalance(asset_returns: pd.DataFrame,
                 weights: dict[str, float] | pd.DataFrame,
                 initial_capital: float,
                 rebalance_days: int) -> pd.Series:
    """Compound a portfolio daily, snapping back to target `weights` every `rebalance_days`.

    Weights are assumed perfectly divisible: no share counts, no transaction costs.

    Between rebalances each holding drifts with its own return; on a rebalance date the
    portfolio value is redistributed across the target weights and compounding resumes.
    Vectorised by splitting the horizon into fixed-length blocks, so there is no day loop.

    Parameters
    ----------
    asset_returns   : pd.DataFrame      daily simple returns; DatetimeIndex rows x ticker
                                        columns, first row 0.0 (inception)
    weights         : dict | DataFrame  either one target weight per ticker, held for the whole
                                        horizon, or per-block weights as returned by
                                        `build_block_weights` — one row per block, ticker
                                        columns. Tickers must be a subset of
                                        `asset_returns.columns`, and each row must sum to 1.0
    initial_capital : float             portfolio value on the first date, in dollars
    rebalance_days  : int               trading days between rebalances

    Returns
    -------
    pd.Series  float, named "Value", indexed like `asset_returns`: the portfolio's dollar
               value on each date, starting at `initial_capital`
    """
    blocks = pd.Series(np.arange(len(asset_returns)) // rebalance_days,
                       index = asset_returns.index)

    # One weight vector per block. A dict is the constant-weight case, broadcast to every block,
    # so both regimes take exactly the same path from here on.
    weights_by_block =\
    (
        pd.DataFrame([weights] * (blocks.iloc[-1] + 1), dtype = float)
        if isinstance(weights, dict) else
        weights.astype(float)
    )

    value_per_dollar =\
    (
        asset_returns
        [weights_by_block.columns]       # only this portfolio's assets, in weight order
        .add(1)
        .groupby(blocks)
        .cumprod()                       # growth of $1 per asset, restarting each block
        .mul(weights_by_block            # at each block's own target weights
             .loc[blocks.to_numpy()]
             .set_axis(asset_returns.index))
        .sum(axis = "columns")           # -> portfolio value per $1 invested at block start
    )

    capital_at_block_start =\
    (
        value_per_dollar
        .groupby(blocks)
        .last()                          # each block's total growth factor
        .shift(1, fill_value = 1.0)      # block N starts with what blocks 0..N-1 earned
        .cumprod()
        .mul(initial_capital)
    )

    return (
        value_per_dollar
        .mul(blocks.map(capital_at_block_start))
        .rename("Value")
    )

In [134]:
def optimise_weights(window_returns: pd.DataFrame,
                     rf_window: pd.Series,
                     trading_days: int,
                     mc_runs: int,
                     rng: np.random.Generator) -> np.ndarray:
    """Pick the long-only mix with the highest Sharpe ratio, by random search.

    This is the Markowitz tangency portfolio — the best return per unit of risk available
    from these assets — approximated by sampling instead of solved in closed form.

    Three steps:

    1. Draw `mc_runs` random weight vectors. Each is non-negative and rescaled to sum to 1,
       so every candidate is a valid long-only, fully-invested portfolio by construction —
       nothing has to be rejected or penalised.
    2. Score them all in one matrix product. `(days x assets) @ (assets x runs)` produces one
       excess-return path per candidate, and two column-wise reductions turn those paths into
       annualised Sharpe ratios. There is no loop over draws.
    3. Return the weights of the best-scoring draw.

    Step 2 uses the same Sharpe formula as `compute_metrics` (`ddof = 1`, annualised by
    `sqrt(trading_days)`), so the quantity maximised here is the quantity section 6.1 reports.

    Being a sample rather than a solve, the answer is approximate: more assets need more draws,
    and the search is biased toward balanced mixes because normalising uniform draws clusters
    them near equal weight rather than spreading them evenly over the simplex.

    Parameters
    ----------
    window_returns : pd.DataFrame        daily simple returns over the estimation window;
                                         date rows x ticker columns
    rf_window      : pd.Series           daily risk-free rate, sharing that index
    trading_days   : int                 annualisation factor (252)
    mc_runs        : int                 number of random allocations to draw
    rng            : np.random.Generator seeded source, so the search is reproducible

    Returns
    -------
    np.ndarray  one weight per column of `window_returns`, in that order; non-negative and
                summing to 1.0

    See Also
    --------
    README.md, section 2 — a worked example with input and output.
    """
    draws  = rng.random((mc_runs, window_returns.shape[1]))
    draws /= draws.sum(axis = 1, keepdims = True)      # long-only and fully invested

    # (days x assets) @ (assets x runs) -> one excess-return series per candidate allocation
    excess =\
    (
        window_returns
        .sub(rf_window, axis = "index")
        .to_numpy()
        @ draws.T
    )

    # Same annualised Sharpe as `compute_metrics`, so the objective maximised here is the
    # number §6.1 reports. ddof = 1 matches pandas' default.
    with np.errstate(invalid = "ignore", divide = "ignore"):
        sharpe = excess.mean(axis = 0) / excess.std(axis = 0, ddof = 1) * np.sqrt(trading_days)

    return draws[np.nanargmax(sharpe)]


def build_block_weights(asset_returns: pd.DataFrame,
                        rf_daily: pd.Series,
                        weights: dict[str, float],
                        rebalance_days: int,
                        optimise: bool,
                        lookback_days: int,
                        trading_days: int,
                        mc_runs: int,
                        rng: np.random.Generator,
                        min_window: int = 20) -> pd.DataFrame:
    """Build the table of weights each rebalance block trades on — one row per block.

    This is the schedule `do_rebalance` consumes. It has two modes:

    - `optimise = False` — every row is the section 2 mix, so the fixed-weight backtest is
      reproduced exactly. Flipping the flag off always gets the original result back.
    - `optimise = True`  — each block is solved separately by `optimise_weights`, on the
      returns that came immediately before it.

    The estimation window for block `b` is the `lookback_days` of returns ending where the
    block starts:

        rows [b * rebalance_days - lookback_days  :  b * rebalance_days]

    That right edge is the important part. The slice stops at the block's own first row, so a
    block is only ever allocated using returns that had already happened when the rebalance was
    placed — it never sees the returns it goes on to earn. Fitting on the full sample instead
    would hand every block the answer key and inflate alpha by construction.

    Two kinds of block fall back to the section 2 mix instead of being optimised: block 0,
    whose window is empty, and any early block whose window is still shorter than `min_window`.
    Note the window only reaches its full `lookback_days` once `b * rebalance_days` exceeds it;
    until then a block estimates on whatever shorter history exists, which is noisier.

    One `rng` is threaded through every block, so a single seed fixes the whole schedule.

    Parameters
    ----------
    asset_returns  : pd.DataFrame        daily simple returns; DatetimeIndex rows x ticker columns
    rf_daily       : pd.Series           daily risk-free rate, sharing that index
    weights        : dict[str, float]    the section 2 target mix — its keys are the investable
                                         universe, its values the fallback allocation
    rebalance_days : int                 trading days between rebalances
    optimise       : bool                `OPTIMISE_WEIGHTS` from section 2
    lookback_days  : int                 longest trailing estimation window
    trading_days   : int                 annualisation factor (252)
    mc_runs        : int                 Monte Carlo draws per rebalance
    rng            : np.random.Generator seeded source, so the whole schedule is reproducible
    min_window     : int                 shortest window worth estimating from

    Returns
    -------
    pd.DataFrame  indexed 0 .. n_blocks-1 by block number, columns ordered like `weights`;
                  every row non-negative and summing to 1.0

    See Also
    --------
    README.md, section 2 — a worked example with input and output.
    """
    tickers  = list(weights)
    n_blocks = -(-len(asset_returns) // rebalance_days)        # ceiling division

    block_weights =\
    (
        pd.DataFrame([weights] * n_blocks, dtype = float)
        .rename_axis("Block")
    )

    if not optimise:
        return block_weights

    for block in range(n_blocks):
        start  = block * rebalance_days                        # first row the block trades
        window = asset_returns[tickers].iloc[max(0, start - lookback_days) : start]

        if len(window) >= min_window:
            block_weights.iloc[block] = optimise_weights(window,
                                                         rf_daily.loc[window.index],
                                                         trading_days,
                                                         mc_runs,
                                                         rng)

    return block_weights

In [135]:
def compute_drawdown(value: pd.Series) -> pd.Series:
    """Percentage decline from the running peak, for every date.

    Parameters
    ----------
    value : pd.Series  a positive level series, e.g. the dollar values from `do_rebalance`

    Returns
    -------
    pd.Series  float, indexed like `value`: 0.0 on a day that sets a new peak, negative
               below it (-0.25 = 25% under water)
    """
    return (
        value
        .div(value.cummax())
        .sub(1)
    )


def compute_cvar(portfolio_returns: pd.Series,
                 confidence: float = 0.95) -> float:
    """Conditional value at risk: the average daily return on the worst `1 - confidence` days.

    Historical (non-parametric): no distribution is assumed, the sample's own left tail *is*
    the estimate. Value at risk is the quantile that cuts the tail off; CVaR (expected
    shortfall) is the mean of everything beyond it, so it answers "when a bad day happens,
    how bad on average" rather than "how bad at the threshold" — and unlike VaR it is
    sensitive to how fat the tail is behind the cutoff.

    Parameters
    ----------
    portfolio_returns : pd.Series  daily simple returns of one portfolio
    confidence        : float      tail cutoff, 0.95 -> the worst 5% of days

    Returns
    -------
    float  negative decimal, same sign convention as a drawdown (-0.03 = the worst 5% of
           days lose 3% on average)
    """
    value_at_risk = portfolio_returns.quantile(1 - confidence)   # left-tail cutoff

    return (
        portfolio_returns
        [portfolio_returns <= value_at_risk]                     # the tail itself
        .mean()
    )

In [136]:
def compute_metrics(portfolio_returns: pd.Series,
                    rf_daily: pd.Series,
                    trading_days: int) -> dict[str, float]:
    """Return / volatility / Sharpe, reported both over the full horizon and annualised.

    Parameters
    ----------
    portfolio_returns : pd.Series  daily simple returns of one portfolio
    rf_daily          : pd.Series  daily risk-free rate, sharing the same index
    trading_days      : int        annualisation factor (252)

    Returns
    -------
    dict[str, float]  six metrics keyed by name: "Horizon Return", "Annualised Return",
                      "Horizon Volatility", "Annualised Volatility", "Horizon Sharpe",
                      "Annualised Sharpe". Returns and volatilities are decimals
                      (0.12 = 12%); the Sharpe ratios are unitless.
    """
    horizon_days   = len(portfolio_returns)
    horizon_return = portfolio_returns.add(1).prod() - 1
    daily_vol      = portfolio_returns.std()

    excess_returns = portfolio_returns.sub(rf_daily)
    annual_sharpe  = excess_returns.mean() / excess_returns.std() * np.sqrt(trading_days)

    return {
        "Horizon Return"       : horizon_return,
        "Annualised Return"    : (1 + horizon_return) ** (trading_days / horizon_days) - 1,
        "Horizon Volatility"   : daily_vol * np.sqrt(horizon_days),
        "Annualised Volatility": daily_vol * np.sqrt(trading_days),
        "Horizon Sharpe"       : annual_sharpe * np.sqrt(horizon_days / trading_days),
        "Annualised Sharpe"    : annual_sharpe,
    }

In [137]:
def compute_alpha_beta(excess_investor: pd.Series,
                       excess_benchmark: pd.Series,
                       trading_days: int,
                       confidence: float = 0.95) -> dict[str, float]:
    """OLS of the investor's excess return on the benchmark's excess return, with inference.

        R_p - R_f = alpha + beta * (R_m - R_f) + epsilon

    beta  -> sensitivity to the benchmark (just the market)
    alpha -> average excess return the benchmark does not explain (skill)

    Tests H_0: alpha = 0 two ways. The textbook standard error assumes i.i.d. homoskedastic
    residuals; daily returns are neither, being volatility-clustered and mildly autocorrelated,
    so a Newey-West HAC standard error is reported alongside it. Where the two disagree, trust
    the HAC one -- the textbook error is the optimistic one.

    Parameters
    ----------
    excess_investor  : pd.Series  daily investor excess returns  (R_p - R_f), the regressand
    excess_benchmark : pd.Series  daily benchmark excess returns (R_m - R_f), the regressor;
                                  same index as `excess_investor`
    trading_days     : int        annualisation factor (252)
    confidence       : float      two-sided confidence level for the alpha interval (0.95)

    Returns
    -------
    dict[str, float]  "Beta" (unitless slope), "Alpha (daily)" (decimal intercept),
                      "Annualised Alpha" (decimal, alpha compounded over `trading_days`),
                      "R-squared" (0.0 to 1.0), then the inference on alpha:
                      "SE Alpha (OLS)" / "t-stat (OLS)" / "p-value (OLS)" under i.i.d. errors,
                      "SE Alpha (HAC)" / "t-stat (HAC)" / "p-value (HAC)" Newey-West corrected,
                      "NW Lags" (Bartlett truncation lag), and the annualised alpha interval
                      "Annual Alpha CI Low" / "Annual Alpha CI High" (decimals, from the HAC SE)
    """
    beta        = excess_investor.cov(excess_benchmark) / excess_benchmark.var()
    alpha_daily = excess_investor.mean() - beta * excess_benchmark.mean()

    # --- residuals and the textbook (i.i.d.) standard error of the intercept ---
    residuals = excess_investor - alpha_daily - beta * excess_benchmark
    n         = len(excess_investor)
    dof       = n - 2                                    # two estimated parameters
    sigma2    = (residuals ** 2).sum() / dof
    x_mean    = excess_benchmark.mean()
    sxx       = ((excess_benchmark - x_mean) ** 2).sum()

    se_ols    = np.sqrt(sigma2 * (1 / n + x_mean ** 2 / sxx))

    # --- Newey-West HAC standard error: sandwich with Bartlett-weighted autocovariances ---
    design    = np.column_stack([np.ones(n), excess_benchmark.to_numpy()])
    scores    = residuals.to_numpy()[:, None] * design   # h_t = e_t * X_t
    nw_lags   = int(4 * (n / 100) ** (2 / 9))            # standard truncation rule

    meat = scores.T @ scores
    for lag in range(1, nw_lags + 1):
        gamma = scores[lag:].T @ scores[:-lag]
        meat += (1 - lag / (nw_lags + 1)) * (gamma + gamma.T)

    bread     = np.linalg.inv(design.T @ design)
    se_hac    = np.sqrt((bread @ meat @ bread)[0, 0])    # [0, 0] is the intercept's variance

    t_ols     = alpha_daily / se_ols
    t_hac     = alpha_daily / se_hac
    t_crit    = stats.t.ppf(0.5 + confidence / 2, dof)

    # Compounding is monotone, so transforming the daily endpoints preserves the coverage.
    ci_daily  = alpha_daily - t_crit * se_hac, alpha_daily + t_crit * se_hac

    return {
        "Beta"                : beta,
        "Alpha (daily)"       : alpha_daily,
        "Annualised Alpha"    : (1 + alpha_daily) ** trading_days - 1,
        "R-squared"           : excess_investor.corr(excess_benchmark) ** 2,
        "SE Alpha (OLS)"      : se_ols,
        "t-stat (OLS)"        : t_ols,
        "p-value (OLS)"       : 2 * stats.t.sf(abs(t_ols), dof),
        "SE Alpha (HAC)"      : se_hac,
        "t-stat (HAC)"        : t_hac,
        "p-value (HAC)"       : 2 * stats.t.sf(abs(t_hac), dof),
        "NW Lags"             : nw_lags,
        "Annual Alpha CI Low" : (1 + ci_daily[0]) ** trading_days - 1,
        "Annual Alpha CI High": (1 + ci_daily[1]) ** trading_days - 1,
    }

---
## 5. Backtest

### 5.1 Portfolio values

Three curves, from two portfolios. All start at `INITIAL_CAPITAL` and compound daily.

The benchmark holds its §2 mix throughout — it is the yardstick. The investor is run **twice from
the same §2 mix**: once rebalancing every `REBALANCE_DAYS`, and once bought on day 0 and never
reset. Everything downstream — returns, excess returns, metrics, drawdowns, the summary table and
the charts — flows from these columns, so the price of the rebalancing schedule sits next to the
result it buys in every one of them, with no switch to flip and re-run.

The rebalanced investor's mix comes from `build_block_weights` — identical to the §2 mix when
`OPTIMISE_WEIGHTS` is off, and re-solved for maximum Sharpe on the preceding `LOOKBACK_DAYS` at
every rebalance when it is on. The held investor takes the §2 mix directly: a single block spanning
the horizon has no preceding window to solve on, so `OPTIMISE_WEIGHTS` cannot reach it either way.

In [138]:
# The three column labels, named once. Every table and chart below is keyed by these, so the
# names cannot drift apart between the summary and the legends.
INVESTOR_REBALANCED = "Investor (rebalanced)"
INVESTOR_HELD       = "Investor (held)"
BENCHMARK           = "Benchmark"

# One weight vector per rebalance block. With OPTIMISE_WEIGHTS off this is the §2 mix repeated,
# so the fixed-weight backtest is reproduced exactly; with it on, each block is solved on its own
# trailing window. The benchmark is always static — it is the yardstick.
investor_weights =\
(
    build_block_weights(asset_returns,
                        rf_daily,
                        INVESTOR_PORTFOLIO,
                        REBALANCE_DAYS,
                        OPTIMISE_WEIGHTS,
                        LOOKBACK_DAYS,
                        TRADING_DAYS,
                        MC_RUNS,
                        np.random.default_rng(RANDOM_SEED))
)

# A rebalance block as long as the whole horizon *is* buy-and-hold: one block, one set of weights,
# no reset. So the held run needs no second function — only HORIZON_DAYS in place of
# REBALANCE_DAYS, and the §2 mix straight from the dict rather than the solved block table.
portfolio_values =\
(
    pd.DataFrame(
        {INVESTOR_REBALANCED: do_rebalance(asset_returns, investor_weights,
                                           INITIAL_CAPITAL, REBALANCE_DAYS),
         INVESTOR_HELD      : do_rebalance(asset_returns, INVESTOR_PORTFOLIO,
                                           INITIAL_CAPITAL, HORIZON_DAYS),
         BENCHMARK          : do_rebalance(asset_returns, BENCHMARK_PORTFOLIO,
                                           INITIAL_CAPITAL, REBALANCE_DAYS)}
    )
    .rename_axis("Date")
)

portfolio_values

,Investor (rebalanced),Investor (held),Benchmark
Date,,,
2016-01-04,"100,000.000000","100,000.000000","100,000.000000"
2016-01-05,"100,423.544155","100,423.544155","100,169.140180"
2016-01-06,"99,476.974068","99,476.974068","98,905.557698"
2016-01-07,"97,479.414515","97,479.414515","96,532.712048"
2016-01-08,"96,453.234537","96,453.234537","95,473.101485"
...,...,...,...
2026-10-12,"372,206.038913","378,131.351167","460,273.223807"
2026-10-13,"372,035.982189","377,987.201757","460,220.005892"
2026-10-14,"369,998.991209","375,470.397040","455,245.177894"


### 5.2 Portfolio returns and excess returns

Rebalancing is value-neutral (it only reshuffles an unchanged total), so the daily return of the
portfolio is simply the day-over-day change in its value, including across rebalance dates.

`excess_returns` carries all three risk-free conventions from §3.3 at once, as a two-level column
index — `("10-Year Treasury", "Investor (rebalanced)")`, `("5-Year Treasury", "Benchmark")`, and so
on — rather than just the one `RISK_FREE_TICKER` selects. §6.3's `alpha_beta` still reads only its
own slice; the other two live here so §6.3's comparison and §7's chart don't have to subtract `rf`
a second time. With three portfolios and three conventions that is nine columns, and §6.3 regresses
six of them.

In [139]:
portfolio_returns =\
(
    portfolio_values
    .pct_change()
    .fillna(0)                           # day 0 = inception
)

portfolio_returns

,Investor (rebalanced),Investor (held),Benchmark
Date,,,
2016-01-04,0.000000,0.000000,0.000000
2016-01-05,0.004235,0.004235,0.001691
2016-01-06,-0.009426,-0.009426,-0.012614
2016-01-07,-0.020081,-0.020081,-0.023991
2016-01-08,-0.010527,-0.010527,-0.010977
...,...,...,...
2026-10-12,0.000699,0.000410,-0.000604
2026-10-13,-0.000457,-0.000381,-0.000116
2026-10-14,-0.005475,-0.006658,-0.010810


In [140]:
excess_returns =\
(
    pd.concat({label: portfolio_returns.sub(rf, axis = "index")
              for label, rf in RF_CONVENTIONS.values()},
             axis  = 1,
             names = ["Risk-free convention", "Portfolio"])
)

excess_returns

Risk-free convention      10-Year Treasury                            \
Portfolio            Investor (rebalanced) Investor (held) Benchmark   
Date                                                                   
2016-01-04                       -0.000088       -0.000088 -0.000088   
2016-01-05                        0.004147        0.004147  0.001603   
2016-01-06                       -0.009511       -0.009511 -0.012700   
2016-01-07                       -0.020165       -0.020165 -0.024076   
2016-01-08                       -0.010611       -0.010611 -0.011060   
...                                    ...             ...       ...   
2026-10-12                        0.000496        0.000207 -0.000806   
2026-10-13                       -0.000659       -0.000583 -0.000317   
2026-10-14                       -0.005676       -0.006859 -0.011010   
2026-10-15                        0.005982        0.005246  0.002649   
2026-10-16                       -0.007249       -0.008133 -0.011262   

Risk-free convention       5-Year Treasury                            \
Portfolio            Investor (rebalanced) Investor (held) Benchmark   
Date                                                                   
2016-01-04                       -0.000068       -0.000068 -0.000068   
2016-01-05                        0.004168        0.004168  0.001624   
2016-01-06                       -0.009491       -0.009491 -0.012679   
2016-01-07                       -0.020144       -0.020144 -0.024054   
2016-01-08                       -0.010589       -0.010589 -0.011039   
...                                    ...             ...       ...   
2026-10-12                        0.000504        0.000215 -0.000798   
2026-10-13                       -0.000651       -0.000575 -0.000310   
2026-10-14                       -0.005669       -0.006852 -0.011003   
2026-10-15                        0.005989        0.005253  0.002656   
2026-10-16                       -0.007242       -0.008126 -0.011255   

Risk-free convention                Rf = 0                            
Portfolio            Investor (rebalanced) Investor (held) Benchmark  
Date                                                                  
2016-01-04                        0.000000        0.000000  0.000000  
2016-01-05                        0.004235        0.004235  0.001691  
2016-01-06                       -0.009426       -0.009426 -0.012614  
2016-01-07                       -0.020081       -0.020081 -0.023991  
2016-01-08                       -0.010527       -0.010527 -0.010977  
...                                    ...             ...       ...  
2026-10-12                        0.000699        0.000410 -0.000604  
2026-10-13                       -0.000457       -0.000381 -0.000116  
2026-10-14                       -0.005475       -0.006658 -0.010810  
2026-10-15                        0.006179        0.005443  0.002846  
2026-10-16                       -0.007052       -0.007936 -0.011065  

[2713 rows x 9 columns]

---
## 6. Result analysis

### 6.1 Return, volatility and Sharpe

In [141]:
metrics_table =\
(
    pd.DataFrame(
        {column: compute_metrics(portfolio_returns[column], rf_daily, TRADING_DAYS)
         for column in portfolio_values.columns}
    )
    .rename_axis("Metric")
)

metrics_table

,Investor (rebalanced),Investor (held),Benchmark
Metric,,,
Horizon Return,2.696597,2.745179,3.514889
Annualised Return,0.129122,0.130492,0.150291
Horizon Volatility,0.508099,0.512386,0.580171
Annualised Volatility,0.154854,0.156161,0.176820
Horizon Sharpe,2.239787,2.250784,2.373934
Annualised Sharpe,0.682625,0.685976,0.723509


### 6.2 Drawdowns and tail risk

Two views of downside: the **max drawdown** is the worst peak-to-trough loss the path ever sustained, a
peak-relative and path-dependent number; **CVaR** is the average loss on the worst
$1 - \text{confidence}$ of individual days, which ignores the path but says how heavy the left tail is.
One bounds the pain of holding on, the other the pain of a single bad day.

CVaR is reported twice: as a percentage, and as that percentage applied to the portfolio's **latest**
value. The dollar figure is what a worst-5% day costs on the balance actually held today, so it grows
with the account rather than staying pinned to `INITIAL_CAPITAL`.

In [142]:
drawdowns =\
(
    portfolio_values
    .apply(compute_drawdown)
)

CVAR_LABEL        = f"CVaR ({CVAR_CONFIDENCE:.0%}, daily)"
CVAR_DOLLAR_LABEL = f"CVaR ({CVAR_CONFIDENCE:.0%}, daily $)"

latest_value =\
(
    portfolio_values
    .iloc[-1]                            # a Series per portfolio, so it aligns on the index
)

cvar =\
(
    portfolio_returns
    .apply(compute_cvar, confidence = CVAR_CONFIDENCE)
)

risk_table =\
(
    pd.DataFrame({"Max Drawdown"     : drawdowns.min(),
                  "Max Drawdown Date": drawdowns.idxmin(),
                  CVAR_LABEL         : cvar,
                  CVAR_DOLLAR_LABEL  : cvar.mul(latest_value)})   # same loss, on today's balance
    .rename_axis("Portfolio")
)

# Both investor runs, side by side: the worst drawdown a rebalancing schedule avoided (or caused)
# is one of the few places its effect shows up as risk rather than return.
for column in [INVESTOR_REBALANCED, INVESTOR_HELD]:
    print(f"{column}: worst loss from a prior peak {drawdowns[column].min():.2%} "
          f"on {drawdowns[column].idxmin():%d %b %Y}; "
          f"{CVAR_CONFIDENCE:.0%} CVaR {risk_table.loc[column, CVAR_LABEL]:.2%} "
          f"(-${abs(risk_table.loc[column, CVAR_DOLLAR_LABEL]):,.0f} "
          f"on its latest value of ${latest_value[column]:,.0f})")

print(f"\nCVaR is the average return on the worst "
      f"{(1 - CVAR_CONFIDENCE) * len(portfolio_returns):.0f} of "
      f"{len(portfolio_returns):,} days.")

risk_table

Investor (rebalanced): worst loss from a prior peak -33.37% on 23 Mar 2020; 95% CVaR -2.35% (-$8,684 on its latest value of $369,660)
Investor (held): worst loss from a prior peak -33.37% on 23 Mar 2020; 95% CVaR -2.37% (-$8,870 on its latest value of $374,518)

CVaR is the average return on the worst 136 of 2,713 days.


,Max Drawdown,Max Drawdown Date,"CVaR (95%, daily)","CVaR (95%, daily $)"
Portfolio,,,,
Investor (rebalanced),-0.333690,2020-03-23,-0.023491,"-8,683.798836"
Investor (held),-0.333689,2020-03-23,-0.023683,"-8,869.804312"
Benchmark,-0.337173,2020-03-23,-0.026953,"-12,169.125294"


### 6.3 Alpha and beta

Investor excess return regressed on benchmark excess return, under the risk-free convention
`RISK_FREE_TICKER` selects — once for the rebalanced investor and once for the held one, against
the same benchmark. The pair is the point: beta says how much of each run is just benchmark
exposure, and alpha what is left over, so the difference between the two columns is what the
rebalancing schedule did to the decomposition.

In [143]:
# An earlier version of this notebook compared rebalancing frequencies by hand: run the backtest,
# paste the printed regression into a comment, change REBALANCE_DAYS, run it again. That block of
# pasted numbers is gone — §5.1 now backtests the held investor alongside the rebalanced one on
# every run, so the "with vs without rebalancing" comparison below is computed from the current
# parameters instead of copied from an older one, and cannot go stale against §2.
#
# To compare *frequencies* (1d vs 60d vs 100d) rather than on-vs-off, REBALANCE_DAYS in §2 is
# still the dial: change it and re-run. The held column is unaffected — it never rebalances.

In [144]:
# Both investor runs against the same benchmark, under RISK_FREE_TICKER's convention.
# `alpha_beta` (the rebalanced run) is the headline: §6.4, §7 and §8 all read it.
alpha_beta      = compute_alpha_beta(excess_returns[RF_LABEL][INVESTOR_REBALANCED],
                                     excess_returns[RF_LABEL][BENCHMARK],
                                     TRADING_DAYS)

alpha_beta_held = compute_alpha_beta(excess_returns[RF_LABEL][INVESTOR_HELD],
                                     excess_returns[RF_LABEL][BENCHMARK],
                                     TRADING_DAYS)

for name, result in [("rebalanced every "f"{REBALANCE_DAYS}d", alpha_beta),
                     ("held, never reset",                     alpha_beta_held)]:
    print(f"=== Investor {name} ===")
    print(f"R_p - R_f = {result['Alpha (daily)']:.6f} "
          f"+ {result['Beta']:.4f} * (R_m - R_f)      "
          f"[daily, R^2 = {result['R-squared']:.4f}, Rf = {RF_LABEL}]")
    print(f"  OLS         SE = {result['SE Alpha (OLS)']:.6f}   "
          f"t = {result['t-stat (OLS)']:6.3f}   p = {result['p-value (OLS)']:.4f}")
    print(f"  Newey-West  SE = {result['SE Alpha (HAC)']:.6f}   "
          f"t = {result['t-stat (HAC)']:6.3f}   p = {result['p-value (HAC)']:.4f}   "
          f"[{result['NW Lags']} lags]")
    verdict = "reject" if result["p-value (HAC)"] < 0.05 else "cannot reject"
    print(f"  At the 5% level we {verdict} H_0: alpha = 0 on the HAC standard error.")
    print(f"  Annualised alpha {result['Annualised Alpha']:.2%}, "
          f"95% CI [{result['Annual Alpha CI Low']:.2%}, "
          f"{result['Annual Alpha CI High']:.2%}]\n")

(
    pd.DataFrame({INVESTOR_REBALANCED: alpha_beta,
                  INVESTOR_HELD      : alpha_beta_held})
    .rename_axis("Metric")
    .assign(Difference = lambda frame: frame[INVESTOR_REBALANCED] - frame[INVESTOR_HELD])
)

=== Investor rebalanced every 60d ===
R_p - R_f = -0.000015 + 0.8555 * (R_m - R_f)      [daily, R^2 = 0.9541, Rf = 10-Year Treasury]
  OLS         SE = 0.000040   t = -0.369   p = 0.7124
  Newey-West  SE = 0.000041   t = -0.365   p = 0.7153   [8 lags]
  At the 5% level we cannot reject H_0: alpha = 0 on the HAC standard error.
  Annualised alpha -0.37%, 95% CI [-2.35%, 1.65%]

=== Investor held, never reset ===
R_p - R_f = -0.000015 + 0.8665 * (R_m - R_f)      [daily, R^2 = 0.9627, Rf = 10-Year Treasury]
  OLS         SE = 0.000037   t = -0.406   p = 0.6850
  Newey-West  SE = 0.000037   t = -0.398   p = 0.6904   [8 lags]
  At the 5% level we cannot reject H_0: alpha = 0 on the HAC standard error.
  Annualised alpha -0.37%, 95% CI [-2.19%, 1.48%]



,Investor (rebalanced),Investor (held),Difference
Metric,,,
Beta,0.855457,0.866539,-0.011082
Alpha (daily),-0.000015,-0.000015,0.000000
Annualised Alpha,-0.003725,-0.003727,0.000003
R-squared,0.954131,0.962700,-0.008570
SE Alpha (OLS),0.000040,0.000037,0.000004
...,...,...,...
t-stat (HAC),-0.364750,-0.398317,0.033568
p-value (HAC),0.715327,0.690428,0.024899
NW Lags,8.000000,8.000000,0.000000


In [145]:
# The same regression across both dimensions at once: three risk-free conventions x two
# rebalancing modes. excess_returns (§5.2) already carries every column this needs, so this is no
# extra subtraction, just a different slice each time. Only `alpha_beta` above (RISK_FREE_TICKER's
# convention, rebalanced) feeds §6.4 and §8; this table, and §7's chart, are comparison only.
alpha_beta_by_rf =\
(
    pd.DataFrame(
        {(label, mode): compute_alpha_beta(excess_returns[label][column],
                                           excess_returns[label][BENCHMARK],
                                           TRADING_DAYS)
         for label, _ in RF_CONVENTIONS.values()
         for mode, column in [("Rebalanced", INVESTOR_REBALANCED),
                              ("Held",       INVESTOR_HELD)]}
    )
    .rename_axis("Metric")
    .rename_axis(columns = ["Risk-free convention", "Rebalancing"])
)

alpha_beta_by_rf.T

Metric                               Beta  Alpha (daily)  Annualised Alpha  \
Risk-free convention Rebalancing                                             
10-Year Treasury     Rebalanced  0.855457      -0.000015         -0.003725   
                     Held        0.866539      -0.000015         -0.003727   
5-Year Treasury      Rebalanced  0.855460      -0.000013         -0.003364   
                     Held        0.866541      -0.000013         -0.003394   
Rf = 0               Rebalanced  0.855451       0.000001          0.000286   
                     Held        0.866536      -0.000000         -0.000025   

Metric                            R-squared  SE Alpha (OLS)  t-stat (OLS)  \
Risk-free convention Rebalancing                                            
10-Year Treasury     Rebalanced    0.954131        0.000040     -0.368702   
                     Held          0.962700        0.000037     -0.405735   
5-Year Treasury      Rebalanced    0.954133        0.000040     -0.332932   
                     Held          0.962703        0.000037     -0.369415   
Rf = 0               Rebalanced    0.954128        0.000040      0.028259   
                     Held          0.962698        0.000037     -0.002697   

Metric                            p-value (OLS)  SE Alpha (HAC)  t-stat (HAC)  \
Risk-free convention Rebalancing                                                
10-Year Treasury     Rebalanced        0.712379        0.000041     -0.364750   
                     Held              0.684969        0.000037     -0.398317   
5-Year Treasury      Rebalanced        0.739211        0.000041     -0.328877   
                     Held              0.711847        0.000037     -0.362145   
Rf = 0               Rebalanced        0.977457        0.000041      0.027454   
                     Held              0.997848        0.000038     -0.002602   

Metric                            p-value (HAC)  NW Lags  Annual Alpha CI Low  \
Risk-free convention Rebalancing                                                
10-Year Treasury     Rebalanced        0.715327 8.000000            -0.023512   
                     Held              0.690428 8.000000            -0.021875   
5-Year Treasury      Rebalanced        0.742274 8.000000            -0.023188   
                     Held              0.717272 8.000000            -0.021574   
Rf = 0               Rebalanced        0.978100 8.000000            -0.019950   
                     Held              0.997924 8.000000            -0.018568   

Metric                            Annual Alpha CI High  
Risk-free convention Rebalancing                        
10-Year Treasury     Rebalanced               0.016462  
                     Held                     0.014756  
5-Year Treasury      Rebalanced               0.016861  
                     Held                     0.015122  
Rf = 0               Rebalanced               0.020939  
                     Held                     0.018867

### 6.4 Summary table

Every computed figure in one place, for all three columns: returns, volatility, Sharpe, max
drawdown and its date, CVaR, then the whole regression — alpha and beta, both standard errors and
their t-statistics and p-values, the Newey-West truncation lag, and the annualised alpha confidence
interval.

Read it across, not down: the two investor columns differ *only* in whether the §2 mix is snapped
back every `REBALANCE_DAYS`, so every gap between them is the rebalancing schedule's doing and
nothing else.

Alpha and beta describe an investor run *relative to* the benchmark, so they are blank in the
benchmark's own column.

In [146]:
summary_table =\
(
    metrics_table
    .T                                                       # portfolios -> rows
    # every risk column, then every key the regression returned. The regression rows are
    # investor-only — the benchmark cannot have an alpha or a beta against itself — and are
    # supplied as a Series so pandas aligns them on the portfolio name rather than on position,
    # leaving the benchmark's cell NaN without it ever having to be spelled out.
    .assign(**{metric: risk_table[metric] for metric in risk_table.columns},
            **{metric: pd.Series({INVESTOR_REBALANCED: alpha_beta[metric],
                                  INVESTOR_HELD      : alpha_beta_held[metric]})
               for metric in alpha_beta})
    .T                                                       # metrics -> rows
    .rename_axis("Metric")
)

summary_table

,Investor (rebalanced),Investor (held),Benchmark
Metric,,,
Horizon Return,2.696597,2.745179,3.514889
Annualised Return,0.129122,0.130492,0.150291
Horizon Volatility,0.508099,0.512386,0.580171
Annualised Volatility,0.154854,0.156161,0.176820
Horizon Sharpe,2.239787,2.250784,2.373934
...,...,...,...
t-stat (HAC),-0.364750,-0.398317,NaN
p-value (HAC),0.715327,0.690428,NaN
NW Lags,8.000000,8.000000,NaN


In [147]:
# Same table, formatted for reading. Each metric carries its own convention: rates as
# percentages, dollar losses with a sign in front of the currency symbol, daily alpha and its
# standard errors too small for a percentage, p-values that would round to a bare 0.0000, and
# the lag count as a plain integer.
percent_metrics = ["Horizon Return", "Annualised Return", "Horizon Volatility",
                   "Annualised Volatility", "Max Drawdown", CVAR_LABEL, "Annualised Alpha",
                   "Annual Alpha CI Low", "Annual Alpha CI High"]
dollar_metrics  = [CVAR_DOLLAR_LABEL]
daily_metrics   = ["Alpha (daily)", "SE Alpha (OLS)", "SE Alpha (HAC)"]
pvalue_metrics  = ["p-value (OLS)", "p-value (HAC)"]
count_metrics   = ["NW Lags"]


def format_metric(value: float, metric: str) -> str:
    """Render one cell of `summary_table`, choosing the convention from its metric name.

    Parameters
    ----------
    value  : float  the cell's value; `NaN` where the metric does not apply to that portfolio
    metric : str    the row label, i.e. which convention to use

    Returns
    -------
    str  the display string, "—" for a metric that does not apply
    """
    if pd.isna(value):
        return "—"

    if metric in percent_metrics:
        return f"{value:.2%}"
    if metric in dollar_metrics:
        return f"-${abs(value):,.0f}" if value < 0 else f"${value:,.0f}"
    if metric == "Max Drawdown Date":
        return f"{value:%d %b %Y}"
    if metric in daily_metrics:
        return f"{value:.6f}"
    if metric in pvalue_metrics:
        return "<0.0001" if value < 1e-4 else f"{value:.4f}"
    if metric in count_metrics:
        return f"{value:,.0f}"

    return f"{value:.4f}"


summary_display =\
(
    summary_table
    .apply(lambda row: row.map(lambda value: format_metric(value, row.name)),
           axis = "columns")
)

summary_display

,Investor (rebalanced),Investor (held),Benchmark
Metric,,,
Horizon Return,269.66%,274.52%,351.49%
Annualised Return,12.91%,13.05%,15.03%
Horizon Volatility,50.81%,51.24%,58.02%
Annualised Volatility,15.49%,15.62%,17.68%
Horizon Sharpe,2.2398,2.2508,2.3739
...,...,...,...
t-stat (HAC),-0.3647,-0.3983,—
p-value (HAC),0.7153,0.6904,—
NW Lags,8,8,—


---
## 7. Charts

In [148]:
portfolio_values_long =\
(
    portfolio_values
    .reset_index()
    .melt(id_vars    = "Date",
          var_name   = "Portfolio",
          value_name = "Value")
)

portfolio_values_long

,Date,Portfolio,Value
0,2016-01-04,Investor (rebalanced),"100,000.000000"
1,2016-01-05,Investor (rebalanced),"100,423.544155"
2,2016-01-06,Investor (rebalanced),"99,476.974068"
3,2016-01-07,Investor (rebalanced),"97,479.414515"
4,2016-01-08,Investor (rebalanced),"96,453.234537"
...,...,...,...
8134,2026-10-12,Benchmark,"460,273.223807"
8135,2026-10-13,Benchmark,"460,220.005892"
8136,2026-10-14,Benchmark,"455,245.177894"
8137,2026-10-15,Benchmark,"456,540.640579"


In [149]:
def describe(portfolio: dict[str, float]) -> str:
    """A weight dict as a one-line label, e.g. "QQQ 33% / DBMF 33% / GLD 33%".

    Parameters
    ----------
    portfolio : dict[str, float]  ticker -> target weight, as declared in §2

    Returns
    -------
    str  the holdings joined with " / ", each weight as a whole percentage
    """
    return " / ".join(f"{ticker} {weight:.0%}" for ticker, weight in portfolio.items())


# What the rebalanced investor portfolio actually is, in one line. Under OPTIMISE_WEIGHTS the mix
# is re-solved at every rebalance, so a fixed "QQQ 33% / DBMF 33% / GLD 33%" label would
# misdescribe that run. The held run is always the plain §2 mix — one block has no window to
# solve on — so it is described literally whatever OPTIMISE_WEIGHTS says.
investor_label = (f"max-Sharpe MC over {', '.join(INVESTOR_PORTFOLIO)} "
                  f"({MC_RUNS:,} draws, {LOOKBACK_DAYS}d lookback)"
                  if OPTIMISE_WEIGHTS else
                  describe(INVESTOR_PORTFOLIO))

# One palette for the whole notebook: the same three colours key the equity curves, the underwater
# plot and §7's regression panels, so a reader learns them once.
PORTFOLIO_COLORS = {INVESTOR_REBALANCED: "red",
                    INVESTOR_HELD      : "darkorange",
                    BENCHMARK          : "blue"}

# Labels are derived from §2, not typed out, so they cannot drift from the weights actually
# backtested. `values` and `labels` are keyed by portfolio name rather than given as positional
# lists: lets-plot orders a discrete scale by first appearance, so positional lists would hand
# each curve another one's colour and label.
equity_plot =\
(
    ggplot(portfolio_values_long,
           aes(x = "Date",
               y = "Value")
          )
    + geom_line(aes(color = "Portfolio"),
                size = 0.8)
    + scale_color_manual(values = PORTFOLIO_COLORS,
                         labels = {INVESTOR_REBALANCED: f"Investor, rebalanced every "
                                                        f"{REBALANCE_DAYS}d: {investor_label}",
                                   INVESTOR_HELD      : f"Investor, held: "
                                                        f"{describe(INVESTOR_PORTFOLIO)}",
                                   BENCHMARK          : f"Benchmark: "
                                                        f"{describe(BENCHMARK_PORTFOLIO)}"},
                         name   = "Portfolio")
    + scale_y_continuous(format = "$,.0f")
    + labs(title    = f"Growth of ${INITIAL_CAPITAL:,.0f}, with and without a "
                      f"{REBALANCE_DAYS}-trading-day rebalance",
           subtitle = f"{prices.index[0]:%d %b %Y} to {prices.index[-1]:%d %b %Y}",
           x        = "",
           y        = "Portfolio Value")
    + ggsize(1000, 500)
    + theme(legend_position = "top")
)

equity_plot

In [150]:
drawdown_plot =\
(
    ggplot(drawdowns.reset_index()
                    .melt(id_vars    = "Date",
                          var_name   = "Portfolio",
                          value_name = "Drawdown"),
           aes(x = "Date",
               y = "Drawdown")
          )
    # Each portfolio is measured from 0, so the areas must overlap: lets-plot's default
    # position for geom_area is 'gstack', which would draw the second portfolio's fill on top
    # of the first and show a combined depth its own line never reaches. With three curves the
    # fill is turned down further -- the two investor runs track each other closely, so it is
    # the lines, not the shading, that separate them.
    + geom_area(aes(fill = "Portfolio"),
                position = "identity",
                alpha    = 0.22)
    + geom_line(aes(color = "Portfolio"),
                size = 0.5)
    + scale_fill_manual(values  = PORTFOLIO_COLORS, name = "Portfolio")
    + scale_color_manual(values = PORTFOLIO_COLORS, name = "Portfolio")
    + scale_y_continuous(format = ".0%")
    + labs(title    = "Underwater plot: decline from the running peak",
           subtitle = f"Investor shown both ways -- rebalanced every {REBALANCE_DAYS} trading "
                      f"days, and bought on day 0 and never reset",
           x        = "",
           y        = "Drawdown")
    + ggsize(1000, 400)
    + theme(legend_position = "top")
)

drawdown_plot

In [151]:
# How the investor's target mix moved over the horizon. These are the weights set *at* each
# rebalance, not the drifted weights mid-block: inside a block each sleeve compounds at its own
# rate, so the realised mix wanders away from the band drawn here until the next reset.
investor_weights_daily =\
(
    investor_weights
    .loc[np.arange(len(asset_returns)) // REBALANCE_DAYS]   # per-block weights -> one row per date
    .set_axis(asset_returns.index)
    .rename_axis("Date")
)

weights_plot =\
(
    ggplot(investor_weights_daily.reset_index()
                                 .melt(id_vars    = "Date",
                                       var_name   = "Ticker",
                                       value_name = "Weight"),
           aes(x = "Date",
               y = "Weight")
          )
    + geom_area(aes(fill = "Ticker"),          # weights sum to 1, so the stack fills 0-100%
                alpha = 0.75)
    + scale_y_continuous(format = ".0%",
                         limits = [0, 1])
    + labs(title    = "Investor target weights at each rebalance",
           subtitle = investor_label,
           x        = "",
           y        = "Weight")
    + ggsize(1000, 350)
    + theme(legend_position = "top")
)

weights_plot

In [152]:
# Correlation of the daily returns of the assets the investor actually holds. Unlike covariance,
# correlation is already unitless and bounded to [-1, 1], so the colour scale can be anchored to
# that fixed range instead of the sample's own min/max — a -1 always reads as fully red here, not
# just "the most negative cell in this particular matrix". The diagonal is always 1 (an asset
# against itself). With a single-ticker INVESTOR_PORTFOLIO this collapses to one tile — it earns
# its keep once section 2 holds a mix, which is also when the optimiser in section 4 has something
# to solve.
investor_assets = list(INVESTOR_PORTFOLIO)

correlation =\
(
    asset_returns
    [investor_assets]
    .corr()
)

# lets-plot pads a degenerate continuous domain (every cell equal to 1, the single-ticker case)
# before it validates declared limits, so the fixed [-1, 1] range only applies once there is
# more than one tile to show.
correlation_limits = None if correlation.size == 1 else [-1, 1]

correlation_long =\
(
    correlation
    .rename_axis(index = "Row", columns = "Column")
    .stack()
    .rename("Correlation")
    .reset_index()
    # The number is the content of a matrix cell, so every cell carries one; the ink flips to
    # white only where the tile is dark enough to swallow black text.
    .assign(Label = lambda frame: frame["Correlation"].map("{:.2f}".format),
            Ink   = lambda frame: np.where(frame["Correlation"].abs() > 0.5, "white", "#0b0b0b"))
)

correlation_plot =\
(
    ggplot(correlation_long,
           aes(x = "Column",
               y = "Row")
          )
    + geom_tile(aes(fill = "Correlation"),
                width  = 0.96,               # a hairline of surface between cells
                height = 0.96)
    + geom_text(aes(label = "Label",
                    color = "Ink"),
                size = 9)
    # Positive and negative correlation are opposite meanings either side of "no linear
    # relationship" at zero, so the fill diverges from a neutral midpoint with equal arms —
    # fixed to [-1, 1] rather than the sample's range, so colour means the same thing across runs.
    + scale_fill_gradient2(low      = "#2a78d6",         # -1: moves opposite
                           mid      = "#f0efec",         #  0: no linear relationship
                           high     = "#e34948",         # +1: moves together
                           midpoint = 0,
                           limits   = correlation_limits,
                           name     = "Correlation")
    + scale_color_identity()                 # "Ink" holds the colour itself, not a category
    + scale_x_discrete(limits = investor_assets)
    + scale_y_discrete(limits = investor_assets[::-1])   # diagonal top-left -> bottom-right
    + coord_fixed()
    + labs(title    = "Correlation of daily returns, investor portfolio assets",
           subtitle = (f"{prices.index[0]:%d %b %Y} to {prices.index[-1]:%d %b %Y}  |  "
                       f"the diagonal is each asset against itself"),
           x        = "",
           y        = "")
    + ggsize(700, 600)
    # One tile carries no scale worth reading.
    + theme(legend_position = "none" if correlation.size == 1 else "right")
)

correlation_plot

In [153]:
# The regression behind alpha and beta, drawn once per risk-free convention as its own panel.
# Overlaying all three fitted lines on one axis (an earlier version of this cell) turned out
# useless: the daily shift in excess return between conventions is a few basis points, so the
# three lines sit on top of each other regardless of colour or dash. A panel each is the fix --
# every panel keeps its own correct x/y pair and its own fitted line.
#
# The rebalanced-vs-held comparison is drawn *inside* each panel instead, as two fitted lines,
# and deliberately not as two point clouds: the two investor runs hold the same assets at the
# same target weights and differ only in the reset, so their daily excess returns correlate at
# ~0.99 and their clouds would sit on top of each other exactly like the three Rf lines did.
# The regression lines are a different matter -- they genuinely separate, because holding lets
# the winner run and drifts beta upward. So: one cloud (the rebalanced run, in grey, for scale)
# and two lines, which is where the comparison actually lives.
excess_returns_long =\
(
    excess_returns
    .stack(level = "Risk-free convention", future_stack = True)
    .reset_index()
    # one row per (Date, convention, mode); Benchmark stays a column because it is the x axis
    # for both modes rather than a series of its own
    .melt(id_vars    = ["Date", "Risk-free convention", BENCHMARK],
          value_vars = [INVESTOR_REBALANCED, INVESTOR_HELD],
          var_name   = "Rebalancing",
          value_name = "Investor")
    .assign(Rebalancing = lambda frame: frame["Rebalancing"]
                                        .map({INVESTOR_REBALANCED: "Rebalanced",
                                              INVESTOR_HELD      : "Held"}))
)

# Six rows -- one fitted line per (convention, mode) -- rather than the same slope and intercept
# broadcast onto all 11,070 rows of excess_returns_long. Both carry "Risk-free convention", which
# is all facet_wrap needs to route each line into its own panel.
fitted_lines =\
(
    alpha_beta_by_rf
    .T[["Beta", "Alpha (daily)"]]
    .reset_index()
)

LINE_COLORS = {"Rebalanced": "red", "Held": "darkorange"}

alpha_beta_plot =\
(
    ggplot(excess_returns_long,
           aes(x = "Benchmark",
               y = "Investor")
          )
    + geom_point(data  = excess_returns_long.query("Rebalancing == 'Rebalanced'"),
                 color = "grey",
                 alpha = 0.20,
                 size  = 1.5)
    + geom_abline(aes(slope     = "Beta",
                      intercept = "Alpha (daily)",
                      color     = "Rebalancing"),
                  data = fitted_lines,
                  size = 1.0)
    + geom_hline(yintercept = 0, color = "black", size = 0.3)
    + geom_vline(xintercept = 0, color = "black", size = 0.3)
    + scale_color_manual(values = LINE_COLORS, name = "Investor run")
    + facet_wrap("Risk-free convention")
    + scale_x_continuous(format = ".1%")
    + scale_y_continuous(format = ".1%")
    + labs(title    = "R_p - R_f  =  \u03b1  +  \u03b2 (R_m - R_f) -- one panel per risk-free "
                      "convention, one line per rebalancing mode",
           subtitle = (f"Points are the rebalanced run. Selected: {RF_LABEL}  |  "
                       f"rebalanced \u03b2 = {alpha_beta['Beta']:.4f}, "
                       f"\u03b1 = {alpha_beta['Annualised Alpha']:.2%} p.a.  |  "
                       f"held \u03b2 = {alpha_beta_held['Beta']:.4f}, "
                       f"\u03b1 = {alpha_beta_held['Annualised Alpha']:.2%} p.a."),
           x        = "Benchmark excess return  (R_m - R_f)",
           y        = "Investor excess return  (R_p - R_f)")
    + ggsize(1300, 480)
    + theme(legend_position = "top")
)

alpha_beta_plot

In [154]:
dashboard =\
(
    gggrid([equity_plot, correlation_plot, drawdown_plot, weights_plot],
           ncol = 2)
    + ggsize(1250, 1000)
)

dashboard

## 8. At a glance

In [155]:
# The whole result in one output. Change the weights or flip OPTIMISE_WEIGHTS in §2, run all,
# then read only this cell:
# every figure and label below is derived, so nothing here ever needs editing.
verdict = "reject" if alpha_beta["p-value (HAC)"] < 0.05 else "cannot reject"

# Escape the currency where it is literally currency — the table and the initial capital — so
# MathJax skips it and reads only the $$...$$ block as maths. Escaping the whole string, as an
# earlier version did, would blank the formula by escaping its delimiters too.
summary_table_md = summary_display.to_markdown().replace("$", r"\$")
initial_capital  = rf"\${INITIAL_CAPITAL:,.0f}"

# What the rebalancing schedule was worth over the horizon, in the currency it was measured in.
rebalance_delta = (portfolio_values[INVESTOR_REBALANCED].iloc[-1]
                   - portfolio_values[INVESTOR_HELD].iloc[-1])
delta_verdict   = ("ahead of" if rebalance_delta >= 0 else "behind")

summary_markdown = rf"""
---

### **Investor** {investor_label} &nbsp; vs &nbsp; **Benchmark** {describe(BENCHMARK_PORTFOLIO)}

{prices.index[0]:%d %b %Y} → {prices.index[-1]:%d %b %Y} · {HORIZON_DAYS:,} trading days · {initial_capital} initial · Rf = {RF_LABEL}

The investor is shown twice: **rebalanced** every {REBALANCE_DAYS} trading days
({(HORIZON_DAYS - 1) // REBALANCE_DAYS:,} rebalances), and **held** from day 0. Rebalancing
finished \${abs(rebalance_delta):,.0f} {delta_verdict} holding.

{summary_table_md}

$$
R_p - R_f \;=\; {alpha_beta['Alpha (daily)']:.6f} \;+\; {alpha_beta['Beta']:.4f}\,(R_m - R_f)
\qquad \text{{rebalanced, daily}},\; R^2 = {alpha_beta['R-squared']:.4f}
$$

$$
R_p - R_f \;=\; {alpha_beta_held['Alpha (daily)']:.6f} \;+\; {alpha_beta_held['Beta']:.4f}\,(R_m - R_f)
\qquad \text{{held, daily}},\; R^2 = {alpha_beta_held['R-squared']:.4f}
$$

#### At the 5% level we **{verdict}** H₀: α = 0 for the rebalanced run, on the Newey-West standard error (p = {alpha_beta['p-value (HAC)']:.4f}).
"""

display(Markdown(summary_markdown))
display(dashboard)


---

### **Investor** SPY 50% / USMV 50% &nbsp; vs &nbsp; **Benchmark** SPY 100%

04 Jan 2016 → 16 Oct 2026 · 2,713 trading days · \$100,000 initial · Rf = 10-Year Treasury

The investor is shown twice: **rebalanced** every 60 trading days
(45 rebalances), and **held** from day 0. Rebalancing
finished \$4,858 behind holding.

| Metric                | Investor (rebalanced)   | Investor (held)   | Benchmark   |
|:----------------------|:------------------------|:------------------|:------------|
| Horizon Return        | 269.66%                 | 274.52%           | 351.49%     |
| Annualised Return     | 12.91%                  | 13.05%            | 15.03%      |
| Horizon Volatility    | 50.81%                  | 51.24%            | 58.02%      |
| Annualised Volatility | 15.49%                  | 15.62%            | 17.68%      |
| Horizon Sharpe        | 2.2398                  | 2.2508            | 2.3739      |
| Annualised Sharpe     | 0.6826                  | 0.6860            | 0.7235      |
| Max Drawdown          | -33.37%                 | -33.37%           | -33.72%     |
| Max Drawdown Date     | 23 Mar 2020             | 23 Mar 2020       | 23 Mar 2020 |
| CVaR (95%, daily)     | -2.35%                  | -2.37%            | -2.70%      |
| CVaR (95%, daily \$)   | -\$8,684                 | -\$8,870           | -\$12,169    |
| Beta                  | 0.8555                  | 0.8665            | —           |
| Alpha (daily)         | -0.000015               | -0.000015         | —           |
| Annualised Alpha      | -0.37%                  | -0.37%            | —           |
| R-squared             | 0.9541                  | 0.9627            | —           |
| SE Alpha (OLS)        | 0.000040                | 0.000037          | —           |
| t-stat (OLS)          | -0.3687                 | -0.4057           | —           |
| p-value (OLS)         | 0.7124                  | 0.6850            | —           |
| SE Alpha (HAC)        | 0.000041                | 0.000037          | —           |
| t-stat (HAC)          | -0.3647                 | -0.3983           | —           |
| p-value (HAC)         | 0.7153                  | 0.6904            | —           |
| NW Lags               | 8                       | 8                 | —           |
| Annual Alpha CI Low   | -2.35%                  | -2.19%            | —           |
| Annual Alpha CI High  | 1.65%                   | 1.48%             | —           |

$$
R_p - R_f \;=\; -0.000015 \;+\; 0.8555\,(R_m - R_f)
\qquad \text{rebalanced, daily},\; R^2 = 0.9541
$$

$$
R_p - R_f \;=\; -0.000015 \;+\; 0.8665\,(R_m - R_f)
\qquad \text{held, daily},\; R^2 = 0.9627
$$

#### At the 5% level we **cannot reject** H₀: α = 0 for the rebalanced run, on the Newey-West standard error (p = 0.7153).
